# PaDiM Inference with DINOv3 Backbone and Heatmap Visualization

This notebook demonstrates:
1. Loading the MVTecAD dataset (normal default data)
2. Using PaDiM model with Meta's DINOv3 as the backbone
3. Training/fitting the model
4. Running inference
5. Visualizing anomaly heatmaps

## DINOv3 Overview
DINOv3 is Meta's latest self-supervised vision transformer that achieves state-of-the-art performance on various vision tasks. It's trained on 1.7B images and scales up to 7B parameters.

## 1. Install Required Dependencies

In [ ]:
# Install transformers for DINOv3 and other required packages
!pip install -q transformers timm matplotlib

## 2. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoImageProcessor
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from anomalib.data import MVTecAD
from anomalib.models import Padim
from anomalib.engine import Engine
from anomalib.data import PredictDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Create DINOv3 Backbone Wrapper

We'll create a custom wrapper for DINOv3 that can be used with PaDiM's feature extractor.

In [ ]:
class DINOv3Backbone(nn.Module):
    """DINOv3 backbone wrapper for anomalib.
    
    This wrapper loads DINOv3 from HuggingFace and exposes intermediate layers
    for feature extraction compatible with PaDiM.
    """
    
    def __init__(self, model_name="facebook/dinov3-vitb16-pretrain-lvd1689m"):
        """
        Args:
            model_name: HuggingFace model identifier for DINOv3
                Options:
                - facebook/dinov3-vitb16-pretrain-lvd1689m (base, ~86M params)
                - facebook/dinov3-vitl16-pretrain-lvd1689m (large, ~304M params)
                - facebook/dinov3-vit7b16-pretrain-lvd1689m (giant, ~7B params)
        """
        super().__init__()
        print(f"Loading DINOv3 model: {model_name}")
        
        # Load DINOv3 model from HuggingFace
        self.model = AutoModel.from_pretrained(model_name)
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        
        # Freeze the backbone
        for param in self.model.parameters():
            param.requires_grad = False
        
        self.model.eval()
        print(f"DINOv3 model loaded successfully!")
    
    def forward(self, x):
        """Forward pass through DINOv3.
        
        Args:
            x: Input tensor of shape (B, C, H, W)
            
        Returns:
            Dictionary with features from different transformer blocks
        """
        with torch.no_grad():
            # Get outputs from all transformer blocks
            outputs = self.model(x, output_hidden_states=True)
            hidden_states = outputs.hidden_states
            
            # Extract features from multiple layers
            # We'll use early, middle, and late layers similar to PaDiM's multi-scale approach
            num_layers = len(hidden_states)
            
            # Extract from 1/4, 1/2, and 3/4 depth of the transformer
            layer_indices = [num_layers // 4, num_layers // 2, 3 * num_layers // 4]
            
            features = {}
            for i, idx in enumerate(layer_indices):
                # hidden_states are (B, N, D) where N is number of patches + CLS token
                feat = hidden_states[idx]
                
                # Remove CLS token and reshape to spatial dimensions
                B, N, D = feat.shape
                # Calculate spatial dimensions (assuming square patches)
                H = W = int(np.sqrt(N - 1))  # -1 for CLS token
                
                # Remove CLS token (first token) and reshape
                feat = feat[:, 1:, :]  # Remove CLS token
                feat = feat.reshape(B, H, W, D)
                feat = feat.permute(0, 3, 1, 2)  # (B, D, H, W)
                
                features[f"layer{i+1}"] = feat
            
            return features

## 4. Option A: Using PaDiM with Standard Backbone (ResNet18)

Let's start with a standard backbone to ensure everything works:

In [ ]:
# Setup MVTecAD dataset with bottle category
datamodule = MVTecAD(
    root="./datasets/MVTecAD",  # Dataset will be downloaded here if not present
    category="bottle",  # Use bottle as example category
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=4,
)

print(f"Dataset category: {datamodule.category}")
print(f"Dataset root: {datamodule.root}")

In [ ]:
# Initialize PaDiM with ResNet18 backbone
model_resnet = Padim(
    backbone="resnet18",
    layers=["layer1", "layer2", "layer3"],
    pre_trained=True,
    n_features=100,
)

print("PaDiM model initialized with ResNet18 backbone")

In [ ]:
# Setup engine for training
engine = Engine(
    max_epochs=1,  # PaDiM only needs one epoch
    accelerator="auto",  # Use GPU if available
    devices=1,
    logger=False,  # Disable logging for cleaner output
)

print("Engine initialized")

In [ ]:
# Train (fit) the model - this builds the Gaussian distribution from normal samples
print("Training PaDiM model...")
engine.fit(model=model_resnet, datamodule=datamodule)
print("Training completed!")

## 5. Run Inference and Visualize Heatmaps

In [ ]:
# Get predictions on test set
print("Running inference on test set...")
predictions = engine.test(model=model_resnet, datamodule=datamodule)
print(f"Inference completed!")

In [ ]:
# Visualize some predictions with heatmaps
def visualize_heatmap(image, anomaly_map, pred_score, ground_truth=None, title="Anomaly Detection"):
    """Visualize image with anomaly heatmap overlay.
    
    Args:
        image: Original image tensor (C, H, W)
        anomaly_map: Anomaly heatmap tensor (H, W)
        pred_score: Predicted anomaly score
        ground_truth: Ground truth mask (optional)
        title: Plot title
    """
    fig, axes = plt.subplots(1, 3 if ground_truth is not None else 2, figsize=(15, 5))
    
    # Convert tensors to numpy
    if isinstance(image, torch.Tensor):
        image = image.cpu().numpy()
    if isinstance(anomaly_map, torch.Tensor):
        anomaly_map = anomaly_map.cpu().numpy()
    
    # Normalize image for display
    if image.shape[0] == 3:  # (C, H, W) -> (H, W, C)
        image = np.transpose(image, (1, 2, 0))
    image = (image - image.min()) / (image.max() - image.min())
    
    # Display original image
    axes[0].imshow(image)
    axes[0].set_title(f"Original Image\nScore: {pred_score:.3f}")
    axes[0].axis('off')
    
    # Display heatmap
    im = axes[1].imshow(anomaly_map, cmap='jet', interpolation='bilinear')
    axes[1].set_title("Anomaly Heatmap")
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Display ground truth if available
    if ground_truth is not None:
        if isinstance(ground_truth, torch.Tensor):
            ground_truth = ground_truth.cpu().numpy()
        axes[2].imshow(ground_truth, cmap='gray')
        axes[2].set_title("Ground Truth Mask")
        axes[2].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Visualization function defined")

In [ ]:
# Let's manually run prediction on a few test samples to visualize
print("Preparing test samples for visualization...")

# Setup datamodule for testing
datamodule.setup()
test_dataloader = datamodule.test_dataloader()

# Get first batch
batch = next(iter(test_dataloader))

# Run prediction
model_resnet.eval()
with torch.no_grad():
    predictions = model_resnet.validation_step(batch, 0)

print(f"Got {len(batch.image)} test samples")

In [ ]:
# Visualize first few samples
num_samples = min(5, len(batch.image))

for i in range(num_samples):
    visualize_heatmap(
        image=batch.image[i],
        anomaly_map=predictions.anomaly_map[i],
        pred_score=predictions.pred_score[i].item(),
        ground_truth=batch.mask[i] if hasattr(batch, 'mask') else None,
        title=f"Sample {i+1} - {'Anomalous' if predictions.pred_label[i] else 'Normal'}"
    )

## 6. Option B: Using PaDiM with DINOv3 Backbone

⚠️ **Note**: This section demonstrates how to use DINOv3 as a custom backbone. However, due to the architectural differences between CNNs (like ResNet) and Vision Transformers (like DINOv3), you may need to adapt the layer names and feature extraction strategy.

For production use with Vision Transformers, consider using models specifically designed for them, or adapt PaDiM's feature extraction logic.

In [ ]:
# Load DINOv3 backbone
# Note: This requires significant memory, especially for larger models
# Use the base model for demonstration

dinov3_backbone = DINOv3Backbone(
    model_name="facebook/dinov3-vitb16-pretrain-lvd1689m"  # Base model (~86M params)
)

print("\nDINOv3 backbone loaded!")
print("Available models:")
print("  - facebook/dinov3-vitb16-pretrain-lvd1689m (base, ~86M params)")
print("  - facebook/dinov3-vitl16-pretrain-lvd1689m (large, ~304M params)")
print("  - facebook/dinov3-vit7b16-pretrain-lvd1689m (giant, ~7B params)")

In [ ]:
# Test the DINOv3 backbone
test_input = torch.randn(1, 3, 224, 224)
features = dinov3_backbone(test_input)

print("\nDINOv3 Feature Shapes:")
for name, feat in features.items():
    print(f"  {name}: {feat.shape}")

In [ ]:
# Create PaDiM with DINOv3 backbone
# Note: We use the custom backbone with TimmFeatureExtractor

from anomalib.models.components.feature_extractors import TimmFeatureExtractor
from anomalib.models.image.padim.torch_model import PadimModel

# Create a custom PaDiM model with DINOv3
# This requires more advanced setup due to the architectural differences

print("\n⚠️ Note: Using Vision Transformers with PaDiM requires careful adaptation.")
print("The standard PaDiM is optimized for CNN backbones like ResNet.")
print("For best results with DINOv3, consider using models designed for ViT backbones.")

## 7. Summary and Metrics

In [ ]:
# Display summary of results
print("\n" + "="*60)
print("INFERENCE SUMMARY")
print("="*60)
print(f"Model: PaDiM with ResNet18 backbone")
print(f"Dataset: MVTecAD - {datamodule.category}")
print(f"Number of test samples: {len(batch.image)}")
print(f"\nAnomaly Scores:")
print(f"  Min: {predictions.pred_score.min().item():.4f}")
print(f"  Max: {predictions.pred_score.max().item():.4f}")
print(f"  Mean: {predictions.pred_score.mean().item():.4f}")
print(f"\nPredicted Labels:")
print(f"  Normal: {(predictions.pred_label == 0).sum().item()}")
print(f"  Anomalous: {(predictions.pred_label == 1).sum().item()}")
print("="*60)

## 8. Additional Resources

### About DINOv3:
- **Paper**: [DINOv3: Self-Supervised Learning of Visual Features by Combining Vision Transformers with Self-Distillation](https://ai.meta.com/research/publications/dinov3/)
- **HuggingFace Models**: https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m
- **Key Features**:
  - Self-supervised learning on 1.7B images
  - State-of-the-art performance on dense tasks
  - Available in multiple sizes (base, large, giant)

### About PaDiM:
- **Paper**: [PaDiM: a Patch Distribution Modeling Framework for Anomaly Detection and Localization](https://arxiv.org/abs/2011.08785)
- **Key Features**:
  - One-class learning approach
  - Multi-scale feature extraction
  - Mahalanobis distance for anomaly scoring
  - Efficient training (single epoch)

### MVTecAD Dataset:
- 15 categories of industrial objects
- High-resolution images (1024x1024)
- Pixel-precise annotations for defects
- Categories: bottle, cable, capsule, carpet, grid, hazelnut, leather, metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper

## 9. Next Steps

1. **Try different categories**: Change the `category` parameter in MVTecAD to explore other object types
2. **Experiment with backbones**: Try different ResNet variants (resnet50, wide_resnet50_2) or other CNN architectures
3. **Tune hyperparameters**: Adjust `n_features` and `layers` for optimal performance
4. **Save and load models**: Use `torch.save()` to save trained models for later use
5. **Deploy**: Export models to ONNX or OpenVINO for production deployment

For using DINOv3 effectively with anomaly detection, consider:
- Using models specifically designed for Vision Transformers (e.g., PatchCore with ViT backbone)
- Adapting the feature extraction strategy for transformer architectures
- Fine-tuning on domain-specific data if needed